# Parse TCP HTML Treatises and Build Vector DB with LangChain and ChromaDB

## Overview
This notebook processes HTML files of music theory treatises and creates a vector database for semantic search using LangChain and ChromaDB.

## Key Features of This Notebook
This notebook now uses an intelligent incremental update approach:

1. CONFIGURATION TRACKING (db_config.json)
   - Tracks embedding model, chunk size, and other settings
   - Only recreates DB when breaking changes are detected
   - Breaking changes: embedding model or chunk size changes

2. FILE CHANGE DETECTION (file_hashes.json)
   - Tracks MD5 hash of each source HTML file
   - Only reprocesses files that have changed
   - Skips unchanged files automatically

3. DOCUMENT ID MANAGEMENT
   - Each chunk gets a unique, deterministic ID
   - Allows updating existing documents without duplicates
   - Format: MD5(source_file_page_number_chunk_index)

4. INCREMENTAL UPDATES
   - When a file changes, old documents are deleted first
   - New documents are added with same IDs if content unchanged
   - Prevents duplicate embeddings

### Common Operations:

```python
# Add or update files
process_html_files()  # Only processes changed files

# Force complete rebuild (rare)
process_html_files(force_reprocess=True)

# View database statistics
get_db_stats()

# Remove a specific file
delete_source_file("filename.html")

# Search the database
results = vector_store.similarity_search("your query here", k=5)

# Search with scores
results = vector_store.similarity_search_with_score("your query", k=5)
```

## Main Steps:
1. **Import Libraries** - Load necessary Python packages
2. **Configure Database** - Set up ChromaDB with intelligent update tracking
3. **Parse HTML** - Extract metadata and text from treatises
4. **Create Embeddings** - Generate vector embeddings using OpenAI
5. **Store & Query** - Save to ChromaDB and explore the database

---

## Step 1: Import Required Libraries and API Key

In [2]:
# Standard library imports
import glob
import hashlib
import json
import os
import shutil
import getpass
from pathlib import Path

# Third-party imports
from bs4 import BeautifulSoup

# LangChain imports
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

/Users/rfreedma/anaconda3/envs/lang/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Prompt for OpenAI API key with password masking
print("Please enter your OpenAI API key:")
openai_api_key = getpass.getpass("API Key: ")

# Set as environment variable
os.environ["OPENAI_API_KEY"] = openai_api_key

# Verify it was set (show only first/last few characters for security)
if openai_api_key:
    masked_key = f"{openai_api_key[:7]}...{openai_api_key[-4:]}"
    print(f"✓ API key set successfully: {masked_key}")
else:
    print("✗ No API key entered")

Please enter your OpenAI API key:
✓ API key set successfully: sk-proj...jXgA


## Step 2. Configure Database

In [ ]:
# Configuration for database schema and settings--these will be passed to all the relevant components below
DB_CONFIG = {
    "version": "1.0",
    "embedding_model": "text-embedding-3-small",
    "chunk_size": 2000,
    "chunk_overlap": 300,
    "collection_name": "English_TCP_Treatises"
}

db_path = Path('./chroma-db_tcp_english')
config_path = db_path / 'db_config.json'

# Check if we need to recreate the database
should_recreate = False

if db_path.exists() and config_path.exists():
    # Load existing config
    with open(config_path, 'r') as f:
        existing_config = json.load(f)
    
    # Check for breaking changes
    if (existing_config.get('embedding_model') != DB_CONFIG['embedding_model'] or
        existing_config.get('chunk_size') != DB_CONFIG['chunk_size']):
        print(f"⚠️  Breaking changes detected:")
        print(f"   Old: {existing_config}")
        print(f"   New: {DB_CONFIG}")
        should_recreate = True
    else:
        print(f"✓ Using existing database - configuration unchanged")
        print(f"  Will perform incremental updates only")
elif not db_path.exists():
    print(f"✓ Creating new database at {db_path}")
    should_recreate = True
else:
    print(f"⚠️  Database exists but no config found - will recreate")
    should_recreate = True

# Delete database only if necessary
if should_recreate and db_path.exists():
    shutil.rmtree(db_path)
    print(f"✓ Deleted existing database at {db_path}")

# Create directory if needed
db_path.mkdir(exist_ok=True)

# Save current configuration
with open(config_path, 'w') as f:
    json.dump(DB_CONFIG, f, indent=2)

# Initialize embeddings
embeddings = OpenAIEmbeddings(model=DB_CONFIG['embedding_model'])

# Initialize Chroma vector store
vector_store = Chroma(
    collection_name=DB_CONFIG['collection_name'],
    embedding_function=embeddings,
    persist_directory=str(db_path)
)

# Configure text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=DB_CONFIG['chunk_size'],
    chunk_overlap=DB_CONFIG['chunk_overlap'],
    length_function=len,
    is_separator_regex=False
)

✓ Creating new database at chroma-db_english


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


## Functions to Extract Text + Metadata and Build Vector Store and DB

In [6]:
def generate_document_id(source_file, page_number, chunk_index):
    """Generate a unique, deterministic ID for each document chunk."""
    id_string = f"{source_file}_{page_number}_chunk_{chunk_index}"
    return hashlib.md5(id_string.encode()).hexdigest()

def get_file_hash(filepath):
    """Get MD5 hash of a file to detect changes."""
    hash_md5 = hashlib.md5()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            hash_md5.update(chunk)
    return hash_md5.hexdigest()

def load_file_hashes():
    """Load previously processed file hashes."""
    hash_file = db_path / 'file_hashes.json'
    if hash_file.exists():
        with open(hash_file, 'r') as f:
            return json.load(f)
    return {}

def save_file_hashes(hashes):
    """Save file hashes to track what's been processed."""
    hash_file = db_path / 'file_hashes.json'
    with open(hash_file, 'w') as f:
        json.dump(hashes, f, indent=2)

def extract_metadata(soup):
    """Extract metadata from the HTML record section."""
    metadata = {
        'title': '',
        'author': '',
        'pub_info': '',
        'citation': ''
    }
    
    # Find the record dl element
    record_dl = soup.find('dl', class_='record')
    if not record_dl:
        print("No record dl found.")
        return metadata
    
    # Extract title
    title_div = record_dl.find('div', {'data-key': 'title'})
    if title_div:
        title_dd = title_div.find('dd')
        if title_dd:
            metadata['title'] = title_dd.get_text(strip=True)
    
    # Extract author
    author_div = record_dl.find('div', {'data-key': 'author'})
    if author_div:
        author_dd = author_div.find('dd')
        if author_dd:
            metadata['author'] = author_dd.get_text(strip=True)
    
    # Extract publication info
    pubinfo_div = record_dl.find('div', {'data-key': 'pubinfo'})
    if pubinfo_div:
        pubinfo_dds = pubinfo_div.find_all('dd')
        pub_parts = [dd.get_text(strip=True) for dd in pubinfo_dds]
        metadata['pub_info'] = ' '.join(pub_parts)
    
    # Extract citation - look specifically in the citation section
    citation_dt = record_dl.find('dt', string='Cite this Item')
    if citation_dt:
        citation_dd = citation_dt.find_next('dd')
        if citation_dd:
            citation_span = citation_dd.find('span')
            if citation_span:
                metadata['citation'] = citation_span.get_text(strip=True)
    
    return metadata

def get_document_title(soup, metadata):
    """Extract the overall document title from HTML or metadata."""
    # Use metadata title if available
    if metadata.get('title'):
        return metadata['title']
    
    # Fallback to HTML title tag
    title_tag = soup.find('title')
    if title_tag and title_tag.text.strip():
        return title_tag.text.strip()
    
    # Try h1 or h2 for document title
    h1 = soup.find('h1')
    if h1:
        return h1.text.strip()
    
    h2 = soup.find('h2')
    if h2:
        return h2.text.strip()
    
    return "Untitled Document"

def extract_pages(soup):
    """Extract individual pages with their metadata from the HTML."""
    pages = []
    
    # Find all article elements that represent pages
    articles = soup.find_all('article', class_='fullview-page')
    
    for article in articles:
        # Extract page metadata from the h3 heading
        page_heading = article.find('h3', class_='js-toc-ignore')
        
        if page_heading:
            page_num = page_heading.get('data-p-num', 'Unknown')
            page_label = page_heading.get('data-heading-label', f'Page {page_num}')
            base = page_heading.get('data-base', '')
        else:
            page_num = 'Unknown'
            page_label = 'Unknown Page'
            base = ''
        
        # Extract text content from this page, removing script/style
        for script in article(["script", "style"]):
            script.decompose()
        
        # Get clean text from the page
        text = article.get_text(separator=' ', strip=True)
        text = ' '.join(text.split())
        
        if text.strip():
            pages.append({
                'text': text,
                'page_number': page_num,
                'page_label': page_label,
                'base': base
            })
    
    return pages

def process_html_files(html_dir='html_source', force_reprocess=False):
    """
    Process all HTML files in the specified directory.
    
    Args:
        html_dir: Directory containing HTML files
        force_reprocess: If True, reprocess all files regardless of changes
    """
    html_files = glob.glob(os.path.join(html_dir, '*.html'))
    
    if not html_files:
        print(f"No HTML files found in {html_dir}")
        return
    
    # Load existing file hashes to detect changes
    existing_hashes = load_file_hashes()
    new_hashes = {}
    
    total_chunks = 0
    total_pages = 0
    files_processed = 0
    files_skipped = 0
    files_updated = 0
    
    for filename in html_files:
        try:
            basename = os.path.basename(filename)
            current_hash = get_file_hash(filename)
            new_hashes[basename] = current_hash
            
            # Skip if file hasn't changed (unless force_reprocess is True)
            if not force_reprocess and basename in existing_hashes:
                if existing_hashes[basename] == current_hash:
                    print(f"⊙ {basename} - No changes, skipping")
                    files_skipped += 1
                    continue
                else:
                    print(f"↻ {basename} - File changed, updating...")
                    files_updated += 1
                    # Delete old documents for this file
                    try:
                        vector_store.delete(where={"source_file": basename})
                        print(f"  Deleted old documents for {basename}")
                    except Exception as e:
                        print(f"  Note: Could not delete old documents: {e}")
            else:
                print(f"+ {basename} - New file, processing...")
            
            with open(filename, 'r', encoding='utf-8') as f:
                html_content = f.read()
            
            # Parse HTML
            soup = BeautifulSoup(html_content, 'html.parser')
            
            # Extract metadata from the record section
            doc_metadata = extract_metadata(soup)
            
            # Get document-level title
            doc_title = get_document_title(soup, doc_metadata)
            
            # Extract pages
            pages = extract_pages(soup)
            
            if not pages:
                print(f"Warning: No pages found in {filename}")
                continue
            
            file_chunks = 0
            chunk_counter = 0
            all_chunk_ids = []
            all_chunks = []
            
            # Process each page separately
            for page in pages:
                # Create metadata for this page, combining document and page metadata
                page_metadata = {
                    "document_title": doc_title,
                    "title": doc_metadata['title'],
                    "author": doc_metadata['author'],
                    "pub_info": doc_metadata['pub_info'],
                    "citation": doc_metadata['citation'],
                    "page_number": page['page_number'],
                    "page_label": page['page_label'],
                    "base": page['base'],
                    "source_file": basename
                }
                
                # Split page text into chunks
                chunks = text_splitter.create_documents(
                    texts=[page['text']],
                    metadatas=[page_metadata]
                )
                
                # Generate IDs for each chunk
                for chunk in chunks:
                    chunk_id = generate_document_id(
                        basename,
                        page['page_number'],
                        chunk_counter
                    )
                    all_chunk_ids.append(chunk_id)
                    all_chunks.append(chunk)
                    chunk_counter += 1
                
                file_chunks += len(chunks)
            
            # Add all chunks for this file at once with IDs
            if all_chunks:
                vector_store.add_documents(
                    documents=all_chunks,
                    ids=all_chunk_ids
                )
            
            total_chunks += file_chunks
            total_pages += len(pages)
            files_processed += 1
            
            print(f'✓ {basename}')
            print(f'  Title: {doc_metadata["title"][:80]}...' if len(doc_metadata["title"]) > 80 else f'  Title: {doc_metadata["title"]}')
            print(f'  Author: {doc_metadata["author"]}')
            print(f'  Citation: {doc_metadata["citation"][:80]}...' if len(doc_metadata["citation"]) > 80 else f'  Citation: {doc_metadata["citation"]}')
            print(f'  Pages: {len(pages)} | Chunks: {file_chunks}')
            
        except Exception as e:
            print(f"✗ Error processing {filename}: {str(e)}")
            import traceback
            traceback.print_exc()
            continue
    
    # Save the new hashes
    save_file_hashes(new_hashes)
    
    print(f'\n{"="*50}')
    print(f'ChromaDB Processing Complete')
    print(f'{"="*50}')
    print(f'Total files: {len(html_files)}')
    print(f'  New/Updated: {files_processed}')
    print(f'  Skipped (unchanged): {files_skipped}')
    print(f'  Changed: {files_updated}')
    print(f'Total pages processed: {total_pages}')
    print(f'Total chunks added: {total_chunks}')

def get_db_stats():
    """Get statistics about the current database."""
    all_docs = vector_store.get()
    
    if not all_docs or 'metadatas' not in all_docs:
        print("Database is empty")
        return
    
    total_docs = len(all_docs['ids'])
    
    # Collect statistics
    authors = set()
    sources = set()
    titles = set()
    
    for metadata in all_docs['metadatas']:
        if metadata:
            if 'author' in metadata:
                authors.add(metadata['author'])
            if 'source_file' in metadata:
                sources.add(metadata['source_file'])
            if 'title' in metadata:
                titles.add(metadata['title'])
    
    print(f"Database Statistics:")
    print(f"  Total chunks: {total_docs}")
    print(f"  Unique authors: {len(authors)}")
    print(f"  Unique sources: {len(sources)}")
    print(f"  Unique titles: {len(titles)}")
    print(f"\nAuthors: {sorted(authors)}")
    
    return {
        'total_docs': total_docs,
        'authors': sorted(authors),
        'sources': sorted(sources),
        'titles': sorted(titles)
    }

def delete_source_file(source_filename):
    """Delete all documents from a specific source file."""
    try:
        vector_store.delete(where={"source_file": source_filename})
        print(f"✓ Deleted all documents from {source_filename}")
        
        # Also remove from file hashes
        hashes = load_file_hashes()
        if source_filename in hashes:
            del hashes[source_filename]
            save_file_hashes(hashes)
            print(f"✓ Removed {source_filename} from tracking")
    except Exception as e:
        print(f"✗ Error deleting documents: {e}")

---

## Step 5: Process HTML Files and Create Vector Database

### What Happens Here:

1. **File Hash Tracking**: Computes MD5 hash of each HTML file to detect changes
2. **Smart Processing**: 
   - ⊙ **Skips** unchanged files
   - ↻ **Updates** modified files (deletes old, adds new)
   - + **Processes** new files
3. **Text Chunking**: Splits long pages into 2000-character chunks with 300-char overlap
4. **ID Generation**: Creates deterministic IDs for each chunk (prevents duplicates)
5. **Embedding Creation**: Sends chunks to OpenAI for vector embedding
6. **Database Storage**: Stores embeddings and metadata in ChromaDB

### Understanding Chunks vs Pages:
- **Page**: A logical division from the original document (marked by page break elements)
- **Chunk**: A piece of text ≤2000 characters for optimal embedding
- One page may create multiple chunks if text is long

In [7]:
# Process HTML files - only processes new or changed files by default
# Use force_reprocess=True to reprocess everything
process_html_files(html_dir='english_sources', force_reprocess=False)

+ bevin_1631.html - New file, processing...
✓ bevin_1631.html
  Title: A briefe and short instruction of the art of musicke to teach how to make discan...
  Author: Bevin, Elway, ca. 1554-1638.
  Citation: "A briefe and short instruction of the art of musicke to teach how to make disca...
  Pages: 64 | Chunks: 64
+ playford_1654.html - New file, processing...
✓ playford_1654.html
  Title: A breefe introduction to the skill of musick for song & violl / by J.P.
  Author: Playford, John, 1623-1686?
  Citation: "A breefe introduction to the skill of musick for song & violl / by J.P." In the...
  Pages: 38 | Chunks: 39
+ le_roy_1574.html - New file, processing...
✓ le_roy_1574.html
  Title: A briefe and plaine instruction to set all musicke of eight diuers tunes in tabl...
  Author: Le Roy, Adrian, ca. 1520-1598.
  Citation: "A briefe and plaine instruction to set all musicke of eight diuers tunes in tab...
  Pages: 184 | Chunks: 197
+ bathe_1596.html - New file, processing...
✓ bathe_1596.

---

## Step 6: Database Management and Statistics

In [8]:
# Get current database statistics
get_db_stats()

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


Database Statistics:
  Total chunks: 1306
  Unique authors: 9
  Unique sources: 9
  Unique titles: 9

Authors: ['Bathe, William, 1564-1614.', 'Bevin, Elway, ca. 1554-1638.', 'Descartes, René, 1596-1650.', 'Le Roy, Adrian, ca. 1520-1598.', 'Morley, Thomas, 1557-1603?', 'Ornithoparchus, Andreas, 16th cent.', 'Playford, John, 1623-1686?', 'Ravenscroft, Thomas, 1592?-1635?', 'Robinson, Thomas, fl. 1589-1609.']


{'total_docs': 1306,
 'authors': ['Bathe, William, 1564-1614.',
  'Bevin, Elway, ca. 1554-1638.',
  'Descartes, René, 1596-1650.',
  'Le Roy, Adrian, ca. 1520-1598.',
  'Morley, Thomas, 1557-1603?',
  'Ornithoparchus, Andreas, 16th cent.',
  'Playford, John, 1623-1686?',
  'Ravenscroft, Thomas, 1592?-1635?',
  'Robinson, Thomas, fl. 1589-1609.'],
 'sources': ['bathe_1596.html',
  'bevin_1631.html',
  'descartes_1653.html',
  'le_roy_1574.html',
  'morley_1596.html',
  'ornithiparcus_1609.html',
  'playford_1654.html',
  'ravenscroft_1614.html',
  'robinson_1609.html'],
 'titles': ['A breefe introduction to the skill of musick for song & violl / by J.P.',
  'A briefe and plaine instruction to set all musicke of eight diuers tunes in tableture for the lute With a briefe instruction how to play on the lute by tablature, to conduct and dispose thy hand vnto the lute, with certaine easie lessons for that purpose. And also a third booke containing diuers new excellent tunes. All first writt

---

## Step 7: Explore Database Contents

These cells demonstrate what's stored in the database and how to access it.

### 7.1: View Sample Metadata

Each chunk in the database has metadata that describes its source.

In [9]:
# Get a sample of documents from the database
sample_docs = vector_store.get(limit=3)

# Display metadata from first 3 chunks
print("=" * 60)
print("SAMPLE METADATA FROM DATABASE")
print("=" * 60)

for i, metadata in enumerate(sample_docs['metadatas'][:3], 1):
    print(f"\n📄 Chunk {i}:")
    print(f"   Title: {metadata.get('title', 'N/A')}")
    print(f"   Author: {metadata.get('author', 'N/A')}")
    print(f"   Page: {metadata.get('page_label', 'N/A')}")
    print(f"   Source File: {metadata.get('source_file', 'N/A')}")
    print(f"   Citation: {metadata.get('citation', 'N/A')[:80]}...")

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


SAMPLE METADATA FROM DATABASE

📄 Chunk 1:
   Title: A briefe and short instruction of the art of musicke to teach how to make discant, of all proportions that are in vse: very necessary for all such as are desirous to attaine to knowledge in the art; and may by practice, if they can sing, soone be able to compose three, foure, and five parts: and also to compose all sorts of canons that are usuall, by these directions of two or three parts in one, upon the plain-song. By Elvvay Bevin.
   Author: Bevin, Elway, ca. 1554-1638.
   Page: Page  [unnumbered]
   Source File: bevin_1631.html
   Citation: "A briefe and short instruction of the art of musicke to teach how to make disca...

📄 Chunk 2:
   Title: A briefe and short instruction of the art of musicke to teach how to make discant, of all proportions that are in vse: very necessary for all such as are desirous to attaine to knowledge in the art; and may by practice, if they can sing, soone be able to compose three, foure, and five parts

### 7.2: View Sample Text Chunks

See what the actual text chunks look like.

In [10]:
# Display text content from sample chunks
print("=" * 60)
print("SAMPLE TEXT CHUNKS")
print("=" * 60)

for i, (doc_text, metadata) in enumerate(zip(sample_docs['documents'][:3], sample_docs['metadatas'][:3]), 1):
    print(f"\n📝 Chunk {i}:")
    print(f"   Source: {metadata.get('title', 'N/A')} (Page {metadata.get('page_label', 'N/A')})")
    print(f"   Length: {len(doc_text)} characters")
    print(f"   Text Preview:")
    print(f"   {'-' * 55}")
    # Show first 300 characters
    preview = doc_text[:300] + "..." if len(doc_text) > 300 else doc_text
    print(f"   {preview}")
    print()

SAMPLE TEXT CHUNKS

📝 Chunk 1:
   Source: A briefe and short instruction of the art of musicke to teach how to make discant, of all proportions that are in vse: very necessary for all such as are desirous to attaine to knowledge in the art; and may by practice, if they can sing, soone be able to compose three, foure, and five parts: and also to compose all sorts of canons that are usuall, by these directions of two or three parts in one, upon the plain-song. By Elvvay Bevin. (Page Page  [unnumbered])
   Length: 29 characters
   Text Preview:
   -------------------------------------------------------
   description Page [unnumbered]


📝 Chunk 2:
   Source: A briefe and short instruction of the art of musicke to teach how to make discant, of all proportions that are in vse: very necessary for all such as are desirous to attaine to knowledge in the art; and may by practice, if they can sing, soone be able to compose three, foure, and five parts: and also to compose all sorts of canons tha

### 7.3: View Sample Vectors (Embeddings)

**What are vectors?** Numerical representations of text meaning. OpenAI's `text-embedding-3-large` creates 3072-dimensional vectors.

**Why vectors?** They enable semantic search - finding text with similar *meaning*, not just matching keywords.

In [11]:
# Display information about the embedding vectors
import numpy as np

if 'embeddings' in sample_docs and sample_docs['embeddings']:
    print("=" * 60)
    print("SAMPLE EMBEDDING VECTORS")
    print("=" * 60)
    
    for i, (embedding, metadata) in enumerate(zip(sample_docs['embeddings'][:3], sample_docs['metadatas'][:3]), 1):
        print(f"\n🔢 Chunk {i} Vector:")
        print(f"   Source: {metadata.get('title', 'N/A')} (Page {metadata.get('page_label', 'N/A')})")
        print(f"   Vector Dimensions: {len(embedding)}")
        print(f"   Vector Type: {type(embedding)}")
        print(f"   First 10 values: {embedding[:10]}")
        print(f"   Vector stats:")
        print(f"      - Min value: {min(embedding):.6f}")
        print(f"      - Max value: {max(embedding):.6f}")
        print(f"      - Mean value: {np.mean(embedding):.6f}")
        print(f"      - Std deviation: {np.std(embedding):.6f}")
else:
    print("Note: Embeddings not included in sample. Use include=['embeddings'] when calling get().")
    print("\nTo retrieve with embeddings:")
    print("sample_with_embeddings = vector_store.get(limit=3, include=['embeddings', 'documents', 'metadatas'])")

Note: Embeddings not included in sample. Use include=['embeddings'] when calling get().

To retrieve with embeddings:
sample_with_embeddings = vector_store.get(limit=3, include=['embeddings', 'documents', 'metadatas'])


### 7.4: Perform a Semantic Search

Demonstrate how to search the database by meaning.

In [12]:
# Example: Search for passages about musical concepts
query = "how to play the lute"

print("=" * 60)
print(f"SEMANTIC SEARCH EXAMPLE")
print("=" * 60)
print(f"\nQuery: '{query}'")
print("\nTop 3 most relevant passages:\n")

# Perform similarity search
results = vector_store.similarity_search(query, k=3)

for i, doc in enumerate(results, 1):
    print(f"{'='*60}")
    print(f"Result {i}:")
    print(f"   Title: {doc.metadata.get('title', 'N/A')}")
    print(f"   Author: {doc.metadata.get('author', 'N/A')}")
    print(f"   Page: {doc.metadata.get('page_label', 'N/A')}")
    print(f"\n   Text excerpt:")
    print(f"   {'-'*55}")
    # Show first 400 characters
    preview = doc.page_content[:400] + "..." if len(doc.page_content) > 400 else doc.page_content
    print(f"   {preview}")
    print()

SEMANTIC SEARCH EXAMPLE

Query: 'how to play the lute'

Top 3 most relevant passages:



Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Result 1:
   Title: A briefe and plaine instruction to set all musicke of eight diuers tunes in tableture for the lute With a briefe instruction how to play on the lute by tablature, to conduct and dispose thy hand vnto the lute, with certaine easie lessons for that purpose. And also a third booke containing diuers new excellent tunes. All first written in French by Adrian Le Roy, and now translated into English by F. Ke. gentleman.
   Author: Le Roy, Adrian, ca. 1520-1598.
   Page: Page  [unnumbered]

   Text excerpt:
   -------------------------------------------------------
   description Page [unnumbered] ❧A brief instruction how to plaie on the Lute by Tablatorie, with certaine easie lessons for the purpose: gathered together: to the greate commoditie and pleasure of the learner of the same. By A.R▪ [illustration]

Result 2:
   Title: A briefe and plaine instruction to set all musicke of eight diuers tunes in tableture for the lute With a briefe instruction how to play on the lute

### 7.5: Search with Similarity Scores

See the actual similarity scores to understand how close the matches are.

In [13]:
# Search with similarity scores (lower distance = more similar)
query = "teaching music to beginners"

print("=" * 60)
print(f"SEMANTIC SEARCH WITH SCORES")
print("=" * 60)
print(f"\nQuery: '{query}'")
print("\nResults ranked by similarity:\n")

# Perform similarity search with scores
results_with_scores = vector_store.similarity_search_with_score(query, k=5)

for i, (doc, score) in enumerate(results_with_scores, 1):
    print(f"{'='*60}")
    print(f"Result {i} - Similarity Score: {score:.4f}")
    print(f"   Title: {doc.metadata.get('title', 'N/A')}")
    print(f"   Author: {doc.metadata.get('author', 'N/A')}")
    print(f"   Page: {doc.metadata.get('page_label', 'N/A')}")
    print(f"\n   Text excerpt (first 250 chars):")
    print(f"   {'-'*55}")
    preview = doc.page_content[:250] + "..." if len(doc.page_content) > 250 else doc.page_content
    print(f"   {preview}")
    print()

print("\n💡 Note: Lower scores indicate higher similarity (distance metric)")
print("   Typical range: 0.0 (identical) to 2.0 (very different)")

SEMANTIC SEARCH WITH SCORES

Query: 'teaching music to beginners'

Results ranked by similarity:

Result 1 - Similarity Score: 1.0805
   Title: New citharen lessons with perfect tunings of the same, from foure course of strings to fourteene course, euen to trie the sharpest teeth of enuie, with lessons of all sortes, and methodicall instructions for all professors and practitioners of the citharen. By Thomas Robinson, student in all the seuen liberall sciences.
   Author: Robinson, Thomas, fl. 1589-1609.
   Page: Page  [unnumbered]

   Text excerpt (first 250 chars):
   -------------------------------------------------------
   description Page [unnumbered] Treble. Meane. Base. Tenor. Master. It is well done of you, you haue Gotten a very good Citharen, for it is both faire and good, true fretted, and easie to play vpon. Now in the name of God let vs beginne, and first mark...

Result 2 - Similarity Score: 1.1027
   Title: New citharen lessons with perfect tunings of the same, from fou